# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata using to_json() method
metadata = dataset.metadata.to_json()
print(f"{metadata.get('name', '')}: {metadata.get('description', '')}")

## 2. Data Overview
Review available record sets, fields, and their corresponding `@id` values.

Below, we inspect all available record sets and the fields (columns) within them using their `@id`s as references.

In [ ]:
# Retrieve all record sets available in the dataset
record_sets = []
if hasattr(dataset, 'record_sets'):
    record_sets = list(dataset.record_sets)
if not record_sets:
    # Try alternative (Croissant v1.0)
    if hasattr(dataset, 'get_record_set_ids'):
        record_sets = dataset.get_record_set_ids()

print('Record sets found:')
for rs in record_sets:
    print(f"- @id: {rs}")

# List fields (`@id`) for each record set
for rs in record_sets:
    print(f"\nFields for record set {rs}:")
    try:
        fields = dataset.fields(record_set=rs)
        for field in fields:
            print(f"  - Field name: {field['name']} (@id: {field['@id']})")
    except Exception as e:
        print(f"  [Could not retrieve fields: {e}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose record sets of interest
# Replace these with the actual `@id`s you found above
record_sets_to_extract = record_sets[:1]  # Extract just the first record set for demonstration

dataframes = {}
for record_set_id in record_sets_to_extract:
    # Load all records from the record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"DataFrame for record set {record_set_id} with columns:\n", df.columns.tolist())
            display(df.head())
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data, or grouping data by key attributes for further analysis.

*Note: Replace the `<numeric_field_id>` and `<group_field_id>` below with appropriate values found above. The code includes guards in case the fields don't exist. This step assumes you have at least one numeric field in the selected record set.*

In [ ]:
# Example: Numeric field exploration
import numpy as np

record_set_id = record_sets_to_extract[0]
df = dataframes.get(record_set_id)
if df is not None and len(df.columns) > 0:
    # Try to find a likely numeric column
    numeric_field = None
    for col in df.columns:
        # Check if column type is numeric or can be converted
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
        # Try conversion
        try:
            df[col] = pd.to_numeric(df[col])
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue

    if numeric_field is not None:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75) if df[numeric_field].dtype in [int, float, np.int64, np.float64] else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical column
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical (string/object) column found for grouping.")
    else:
        print("No numeric column found for analysis.")
else:
    print("No DataFrame data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using common plotting libraries. You may need to adjust the field names depending on your data.

In [ ]:
import matplotlib.pyplot as plt

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # Pairwise scatter plot if at least 2 numeric columns exist
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_cols) >= 2:
        pd.plotting.scatter_matrix(df[numeric_cols], figsize=(10,8))
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides protein measurements and associated analysis values from mass spectrometry on extracellular vesicles from human mast cells.
- Using the Croissant metadata, we explored available record sets and their fields by `@id`.
- We loaded the data for inspection and performed simple filtering, normalization, grouping, and visualizations on numeric fields.
- To further analyze other record sets or perform advanced modeling, repeat these steps substituting the appropriate `@id`s.